# Day 40 — Practice Questions

## Random Forests

**Boston University · Data Science Program**

---

15 questions in four parts.

| part | topic | questions |
|---|---|---|
| **A** | Bootstrap sampling and the vote | 4 |
| **B** | What makes the trees differ | 4 |
| **C** | Honest measurement | 4 |
| **D** | Importances, and the limits | 3 |

Run the setup cell first. Several questions use repeated cross-validation and take a minute or two — that is expected.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, cross_validate, train_test_split)

CV  = StratifiedKFold(5, shuffle=True, random_state=0)
RCV = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

# ---- the same 1200 scholarship decisions as the lesson ------------------
rng = np.random.default_rng(4010)
N = 1200
marks      = rng.normal(70, 14, N).clip(35, 100).round(0)
attendance = rng.normal(82, 12, N).clip(40, 100).round(0)
income     = rng.normal(380000, 210000, N).clip(40000, 1500000).round(0)
sport      = rng.binomial(1, 0.25, N).astype(float)
essay      = rng.normal(6.5, 1.8, N).clip(1, 10).round(1)
interview  = rng.normal(6.0, 2.0, N).clip(1, 10).round(1)
siblings   = rng.integers(0, 5, N).astype(float)
distance   = rng.normal(30, 22, N).clip(1, 150).round(0)
first_gen  = rng.binomial(1, 0.35, N).astype(float)
club_hours = rng.normal(4, 3, N).clip(0, 20).round(1)
acad   = ((marks >= 75) & (attendance >= 80)) | (marks >= 88)
merit  = (essay + interview >= 14.5) & (marks >= 65)
spor   = (sport == 1) & (marks >= 62) & (attendance >= 75)
equity = (first_gen == 1) & (marks >= 68) & (income <= 250000)
rule = (acad | merit | spor | equity) & (income <= 700000)
flip = rng.uniform(0, 1, N) < 0.10
awarded = np.where(flip, 1 - rule.astype(int), rule.astype(int))

sch = pd.DataFrame({"marks": marks, "attendance": attendance, "family_income": income,
                    "sport": sport, "essay": essay, "interview": interview,
                    "siblings": siblings, "distance_km": distance,
                    "first_gen": first_gen, "club_hours": club_hours,
                    "awarded": awarded})
X = sch.drop(columns="awarded")
y = sch["awarded"]

NOISE = ["siblings", "distance_km", "club_hours"]     # pure noise, by construction
TUNED = dict(max_depth=6, min_samples_leaf=10, random_state=0)   # the Day 40 tuned tree

print("setup ok —", len(sch), "rows,", X.shape[1], "columns, baseline",
      round(1 - y.mean(), 4))

---
# Part A — Bootstrap sampling and the vote

### A1
Draw one bootstrap sample of the 1200 rows using
`np.random.default_rng(40).integers(0, 1200, 1200)`.

1. How many **distinct** rows does it contain?
2. What fraction is that, and how close is it to the theoretical $1 - 1/e$?
3. How many rows were left out entirely, and what are those rows called?

In [ ]:
# your code here

### A2
Fit random forests with `n_estimators` of 1, 5, 25, 100 and 500 (all `random_state=0`)
and report 5-fold CV for each.

Then compare the shape of that curve with `max_depth` on a **single** tree
(try 2, 4, 6, 10, `None`). The two knobs behave completely differently — say how, and why.

In [ ]:
# your code here

### A3
Fit a 100-tree forest on all the data and get `predict_proba`.

1. For a forest, what does `predict_proba` actually count?
2. How many applicants are decided **unanimously** (all 100 trees agreeing each way)?
3. Find the single most divided applicant. What was the vote, and was the forest right?
4. Why is this something a single decision tree cannot give you?

In [ ]:
# your code here

### A4
Day 38 showed k-NN collapsing when a column's units changed; Day 39 showed a single tree
completely unaffected.

Predict what happens to a **forest**, then test it: divide `family_income` by 100000, and
separately multiply `marks` by 1000. Report CV for all three versions.

In [ ]:
# your code here

---
# Part B — What makes the trees differ

### B1
A random forest uses **two** sources of randomness. Bagging uses only one.

Fit a `BaggingClassifier` of 200 unlimited trees and a `RandomForestClassifier` of 200
trees, and compare 5-fold CV.

Name the randomness the forest adds, and explain *why* it is supposed to help — even
though on this data the result may surprise you.

In [ ]:
# your code here

### B2
Settle B1 properly. Sweep `max_features` over `[1, 2, 3, 5, 7, None]` with 200 trees,
using `RCV` (5-fold × 10 repeats = 50 fits each). Report the mean **and the standard
deviation**.

Which settings, if any, are *established* as better than the default of 3?

In [ ]:
# your code here

### B3
Fit a forest with `oob_score=True` at `n_estimators` of 5, 10, 20, 50, 100 and 300.

For each, print the OOB score next to a proper 5-fold CV score. Is OOB optimistic or
pessimistic — and does the answer depend on the number of trees? Explain the pattern.

In [ ]:
# your code here

### B4
A forest grows unlimited-depth trees by default, which on Day 39 would have been reckless.

Fit forests with `max_depth` of 3, 5, 8 and `None` and report the training score, CV score
and the gap for each.

Then answer: should you prune a forest? Be careful — look at `max_depth=8` under the
plain 5-fold `CV` first, then check it with `RCV`.

In [ ]:
# your code here

---
# Part C — Honest measurement

### C1
Report training score, CV score and the gap for three models: an unlimited single tree, a
tuned single tree (`**TUNED`), and a 100-tree forest.

The forest's gap is enormous. On Day 39 that was the alarm bell for overfitting. Explain
in your own words why it is not a problem here — and what you should look at instead.

In [ ]:
# your code here

### C2
Repeat the lesson's stability test for both models.

Refit on a random 90% of the rows five times (seeds 0–4), ask each fitted model about the
same first 200 applicants, and count how many applicants receive **two different answers**
across the five refits.

Report both counts. Which difference matters more to a scholarship committee — this one,
or the 1.8 points of accuracy?

In [ ]:
# your code here

### C3
Is the forest's 1.8-point win over the tuned tree real, or is it one lucky split?

Score both models over `RCV` (50 folds). Report each mean, the mean of the **paired
difference**, and in how many of the 50 folds the forest won.

Also compare the tuned tree's 50-fold average with the 0.8200 it got from the single
5-fold split, and say what that teaches you.

In [ ]:
# your code here

### C4
A classmate proposes this workflow: *"OOB is free, so I'll use `oob_score_` to pick
`n_estimators` — I'll take whichever number of trees gives the best OOB."*

Using your B3 table, explain exactly what goes wrong, and what they should do instead.

In [ ]:
# your code here

---
# Part D — Importances, and the limits

### D1
Three of the ten columns — `NOISE` — are pure noise by construction.

1. Fit a 300-tree forest on all the data and print `feature_importances_`. Where does
   `club_hours` rank, and what do the three noise columns total?
2. Now split 70/30, fit on the training part, and compute `permutation_importance` on the
   **test** part with `n_repeats=30`. What do the three noise columns total now?
3. Explain why the two disagree so badly.

In [ ]:
# your code here

### D2
On Day 39, duplicating the `marks` column split its importance almost exactly in two
(0.5306 → 0.0906 + 0.4399), summing back to the original.

Do the same to a **forest**: add `marks_copy` as an exact duplicate of `marks`, refit a
300-tree forest, and compare importances before and after. Also check what happens to the
CV score.

Does the forest split the credit the same way a single tree did? Should it?

In [ ]:
# your code here

### D3
Build a dataset where the truth is a **smooth** relationship rather than a set of
thresholds, using the code below (it is Day 39's exam data).

Compare the baseline, logistic regression, a depth-4 tree and a 300-tree forest.

Where does the forest finish, and why can't adding more trees fix it? What does that tell
you about what ensembling does and does not repair?

In [ ]:
r2 = np.random.default_rng(777)
M = 900
hours = r2.normal(12, 4, M).clip(0, 30).round(1)
prev  = r2.normal(60, 15, M).clip(10, 100).round(0)
passed = (r2.uniform(0, 1, M) <
          1 / (1 + np.exp(-(-7.5 + 0.30 * hours + 0.075 * prev)))).astype(int)
XL = pd.DataFrame({"hours": hours, "prev_score": prev})
yL = pd.Series(passed)
print("pass rate:", round(yL.mean(), 4))

In [ ]:
# your code here

---
### End of questions

Check your answers against Day40_Practice_Solutions.ipynb once it is released. If a number does not match, re-run the setup cell first — every question depends on the same 1200 rows.